# Chess Vision - Train Models on Colab GPU

This notebook trains the occupancy and piece classification models on a free Colab GPU.

**Steps:**
1. Clone the repo
2. Download and process the chesscog dataset
3. Train both models (10 epochs, full data)
4. Download the trained ONNX models

**Runtime:** Select GPU in Runtime > Change runtime type > T4 GPU

In [ ]:
# 1. Clone repo and install dependencies
!git clone https://github.com/samegardner/chess-vision.git
%cd chess-vision
!pip install -e ".[dev]" -q

In [ ]:
# 2. Download chesscog dataset from OSF
!pip install osfclient -q
!python scripts/download_data.py --dataset chesscog

In [ ]:
# 2b. Fix split zip if needed (train.zip + train.z01)
import os
chesscog_dir = "data/chesscog"
if os.path.exists(f"{chesscog_dir}/train.z01"):
    !cd {chesscog_dir} && zip -s 0 train.zip --out train_full.zip
    !cd {chesscog_dir} && unzip -q train_full.zip
    !rm {chesscog_dir}/train_full.zip {chesscog_dir}/train.zip {chesscog_dir}/train.z01
    print("Train set extracted.")

# Count samples
for split in ["train", "val", "test"]:
    import glob
    n = len(glob.glob(f"{chesscog_dir}/{split}/*.json"))
    print(f"{split}: {n} samples")

In [ ]:
# 3. Process into per-square crops
!python scripts/download_data.py --dataset chesscog --process-only

In [ ]:
# 4. Train both models (full data, 10 epochs, GPU)
# This should take ~30-60 min on a T4 GPU
!chess-vision train --stage base --epochs 10 --batch-size 128

In [ ]:
# 5. Check results
import os
for f in ["models/occupancy.onnx", "models/piece.onnx"]:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / 1024 / 1024
        print(f"{f}: {size_mb:.1f} MB")
    else:
        print(f"{f}: NOT FOUND")

# Show checkpoint accuracy
for f in ["models/checkpoints/occupancy_best.pt", "models/checkpoints/piece_best.pt"]:
    if os.path.exists(f):
        print(f"{f}: exists")

In [ ]:
# 6. Download trained models
from google.colab import files
files.download('models/occupancy.onnx')
files.download('models/piece.onnx')
files.download('models/checkpoints/occupancy_best.pt')
files.download('models/checkpoints/piece_best.pt')